In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib widget

In [ ]:
import sys
import logging
from pathlib import Path

In [ ]:
import numpy as np

In [ ]:
from ap_model_training import lr
from ap_model_training.losses import loss_creation_functions

In [ ]:
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S'"
    )
)
_logger = logging.getLogger("ap_model_traning")
for _ in _logger.handlers:
    _logger.removeHandler(_)
_logger.setLevel(logging.DEBUG)
_logger.addHandler(_handler)


In [ ]:
all_files_csv = (
    "/ceph/groups/structbio/adaptive_milling_project/2024labels_new/all_files4.csv"
)


In [ ]:
dist_matrix = np.asarray(
    [
        [0.0, 0.1, 0.1, 0.1, 0.1, 0.1],  # padding
        [0.1, 0.0, 0.5, 0.8, 0.6, 0.2],  # background
        [0.1, 0.5, 0.0, 0.8, 0.9, 1.0],  # lamella
        [0.1, 0.8, 0.8, 0.0, 0.7, 0.7],  # GIS
        [0.1, 0.6, 0.9, 0.7, 0.0, 0.9],  # crack
        [0.1, 0.2, 1.0, 0.7, 0.9, 0.0],  # void
    ],
    dtype=np.float32,
)

lr_dir = Path.home() / "ap_model_training" / "learning_rate"
lr_dir.mkdir(exist_ok=True)


for loss_name in loss_creation_functions.keys():
    try:
        lr.plot_learning_rates(
            all_files_csv,
            output_dir=lr_dir,
            cpu_only=False,
            models_to_ignore=["segresnetds2", "segresnetvae"],
            loss_name=loss_name,
            iterations=100,
            image_size=768,
            model_kwargs={"pretrained": True},
            loss_kwargs={
                "weights": (1.0, 4.0, 3.0, 6.0, 2.0),
                "dist_matrix": dist_matrix,
            },
            gpu_number=1,
        )
    except Exception:
        _logger.error(
            "Failed to plot learning rates for loss %s", loss_name, exc_info=True
        )